In [1]:
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 7.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 22.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.0
    Uninstalling datasets-5.0.0:
      Successfully uninstalled datasets-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 16.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
import torch

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, GRPOConfig, GRPOTrainer
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import TextStreamer
from huggingface_hub import login

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
SEED = 42
REPOSITORY_ID = "hendrimardani/fine-tuned-model-indonesian"
REPOSITORY_ID2 = "hendrimardani/fine-tuned-model-indonesian-2"
DATASET = "Ichsan2895/alpaca-gpt4-indonesian"
MODEL_NAME = "unsloth/Llama-3.1-8B"
CHAT_TEMPLATE = "llama-3.1"

In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# from google.colab import userdata
# hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)

In [6]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.562 GB.


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    dtype = None
)

==((====))==  Unsloth 2026.11.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


In [8]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = CHAT_TEMPLATE
)

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing="unsloth"
)

Unsloth 2026.11.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [10]:
dataset = load_dataset(DATASET, split = "train")
print(f"Jumlah data training: {len(dataset)}")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data training: 49969


In [11]:
dataset[0]

{'Unnamed: 0': 1,
 'input': 'Saranlah slogan untuk kampanye daur ulang\n',
 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}

In [12]:
system_prompt = """
    Anda adalah asisten yang cerdas, membantu, dan mendalam.
    Untuk setiap pertanyaan pengguna, Anda harus berpikir selangkah demi selangkah terlebih dahulu dan menampilkan proses penalaran Anda di dalam tag <think>...</think>.
    Setelah menutup tag </think>, Anda harus memberikan jawaban akhir yang jelas dan ringkas.

    Aturan format ketat:
    Output Anda harus mengikuti templat persis seperti ini:
    <think>
    [Proses penalaran dan analisis langkah-demi-langkah Anda di sini]
    </think>
    [Jawaban akhir Anda di sini]
"""

def formatting_prompts_func(examples):
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []


    for inp, out in zip(inputs, outputs):
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": out}
        ]
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }


dataset = dataset.map(formatting_prompts_func, batched=True)
dataset

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'input', 'output', 'text'],
    num_rows: 49969
})

In [13]:
dataset[0]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n\n    Anda adalah asisten yang cerdas, membantu, dan mendalam.\n    Untuk setiap pertanyaan pengguna, Anda harus berpikir selangkah demi selangkah terlebih dahulu dan menampilkan proses penalaran Anda di dalam tag <think>...</think>.\n    Setelah menutup tag </think>, Anda harus memberikan jawaban akhir yang jelas dan ringkas.\n\n    Aturan format ketat:\n    Output Anda harus mengikuti templat persis seperti ini:\n    <think>\n    [Proses penalaran dan analisis langkah-demi-langkah Anda di sini]\n    </think>\n    [Jawaban akhir Anda di sini]\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSaranlah slogan untuk kampanye daur ulang\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sa

In [14]:
raw_dataset = dataset.train_test_split(test_size=0.1, shuffle=True, seed=SEED)
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'input', 'output', 'text'],
        num_rows: 44972
    })
    test: Dataset({
        features: ['Unnamed: 0', 'input', 'output', 'text'],
        num_rows: 4997
    })
})

In [15]:
train_dataset = raw_dataset["train"]
eval_dataset = raw_dataset["test"]

# Percobaan 1

In [16]:
training_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=800,
    learning_rate=2e-4,
    weight_decay=0.0,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    output_dir="outputs",
    logging_steps=10,              # Menampilkan metrik training setiap 10 steps
    eval_strategy="steps",
    eval_steps=50,                 # Menjalankan evaluasi data validasi setiap 50 steps
    per_device_eval_batch_size=4,  # Batch size khusus proses evaluasi
    push_to_hub=True,
    hub_model_id=REPOSITORY_ID,
    hub_strategy="checkpoint",  # Mengizinkan upload checkpoint sebagai folder terpisah ke Hub
    save_strategy="steps",           # Menyimpan berdasarkan jumlah step
    save_steps=50,                   # Simpan setiap 50 steps
    report_to="tensorboard"
)

In [17]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args = training_args,
)

try:
    from huggingface_hub import snapshot_download
    repo_dir = snapshot_download(repo_id=REPOSITORY_ID)
    checkpoint_path = os.path.join(repo_dir, "last-checkpoint")
    trainer.train(resume_from_checkpoint=checkpoint_path)
except:
    trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/44972 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4997 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


.gitattributes: 0.00B [00:00, ?B/s]

... (26 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


In [18]:
model.push_to_hub_merged(
    repo_id=REPOSITORY_ID,
    tokenizer=tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


... (20 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:02<00:00, 60.54s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/hendrimardani/fine-tuned-model-indonesian`


# Percobaan 2

In [19]:
# Ubah hyperparameter learning rate dan weight decay dan menambahkan hyperparameter metric_for_best_model greater_is_better dan max_grad_norm
training_args2 = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=3,
    max_steps=800,
    learning_rate=2e-5,
    weight_decay=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    output_dir="outputs2",
    logging_steps=10,              # Menampilkan metrik training setiap 10 steps
    eval_strategy="steps",
    eval_steps=50,                 # Menjalankan evaluasi data validasi setiap 50 steps
    per_device_eval_batch_size=4,  # Batch size khusus proses evaluasi
    push_to_hub=True,
    hub_model_id=REPOSITORY_ID2,
    hub_strategy="checkpoint",  # Mengizinkan upload checkpoint sebagai folder terpisah ke Hub
    save_strategy="steps",           # Menyimpan berdasarkan jumlah step
    save_steps=50,                   # Simpan setiap 50 steps
    metric_for_best_model="eval_loss",
    greater_is_better=False,       # Set False karena kita ingin loss yang makin kecil/rendah
    max_grad_norm=1.0,
    report_to="tensorboard"
)

In [20]:
trainer2 = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args2,
)

try:
    from huggingface_hub import snapshot_download
    repo_dir = snapshot_download(repo_id=REPOSITORY_ID2)
    checkpoint_path = os.path.join(repo_dir, "last-checkpoint")
    trainer2.train(resume_from_checkpoint=checkpoint_path)
except:
    trainer2.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.
Unsloth: We found double BOS tokens - we shall remove one automatically.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

... (26 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44,972 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss


In [21]:
model.push_to_hub_merged(
    REPOSITORY_ID2,
    tokenizer,
    save_method = "merged_16bit",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

... (18 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:05<00:00, 61.38s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/hendrimardani/fine-tuned-model-indonesian-2`


In [22]:
!pip freeze > requirements.txt